In [ ]:
import sys

if "google.colab" in sys.modules:
    # If running in Google Colab
    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')

    # Install dependancies
    !pip install pymatgen torch_geometric tqdm

    # Set project path
    PROJECT_PATH = "/content/drive/MyDrive/Project"

else:
    PROJECT_PATH = "Project"

Mounted at /content/drive


In [ ]:
# import dependencies
import os
import torch
import torch_geometric
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool, GATConv 
from torch_geometric.loader import DataLoader

from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, f1_score

from train_test_functions import train_gnn_model, evaluate_gnn_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Get the graphical data
train_set = torch.load(os.path.join(PROJECT_PATH, "train.pt"))
val_set = torch.load(os.path.join(PROJECT_PATH, "val.pt"))
test_set = torch.load(os.path.join(PROJECT_PATH, "test.pt"))

# Create data loaders
train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader = DataLoader(val_set, batch_size=32)
test_loader = DataLoader(test_set, batch_size=32)

In [ ]:
# Definite dims
node_dim = train_set[0].x.shape[1]
edge_dim = train_set[0].edge_attr.shape[1]
global_dim = train_set[0].u.shape[0]

# Model 1: CharlesCGCNN

In [ ]:
# CharlesCGCNN Model

class CharlesCGCNN(nn.Module):
    def __init__(self, node_dim, edge_dim):
        super().__init__()

        self.conv1 = GCNConv(node_dim, 128)
        self.conv2 = GCNConv(128, 128)

        self.fc = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 2)
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index).relu()

        x = global_mean_pool(x, batch)
        return self.fc(x)
    
ch_CGCNN = CharlesCGCNN(node_dim=node_dim, edge_dim=edge_dim).to(device)
loss_fn = nn.CrossEntropyLoss()

ch_CGCNN_metrics = train_gnn_model(ch_CGCNN, train_loader, val_loader, loss_fn, epochs=50)
test_acc, test_f1, test_auc = evaluate_gnn_model(ch_CGCNN, test_loader)
ch_CGCNN_test_metrics = {
    "test_acc": test_acc,
    "test_f1": test_f1,
    "test_auc": test_auc
}
print(f"\nTesting {ch_CGCNN.__class__.__name__} on test set...\nTest Accuracy: {test_acc:.4f} | Test F1: {test_f1:.4f} | Test AUC: {test_auc:.4f}")

## Model 2: Attention-based CharlesCGCNN

In [ ]:
class CharlesCGCNN_Attention(nn.Module):
    def __init__(self, node_dim):
        super().__init__()

        # Attention layers
        self.gat1 = GATConv(node_dim, 128, heads=4, concat=True)
        self.gat2 = GATConv(128*4, 128, heads=1, concat=True)

        self.fc = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 2)
        )

    def forward(self, data, return_attention=False):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        # First attention layer
        x, attn1 = self.gat1(x, edge_index, return_attention_weights=True)
        x = x.relu()

        # Second attention layer
        x, attn2 = self.gat2(x, edge_index, return_attention_weights=True)
        x = x.relu()

        pooled = global_mean_pool(x, batch)
        out = self.fc(pooled)

        if return_attention:
            return out, (attn1, attn2)
        return out
    

ch_CGCNN_att = CharlesCGCNN_Attention(node_dim=node_dim).to(device)
loss_fn = nn.CrossEntropyLoss()

ch_CGCNN_att_metrics = train_model(ch_CGCNN_att, train_loader, val_loader, loss_fn, epochs=50)
test_acc, test_f1, test_auc = evaluate_model(ch_CGCNN_att, test_loader)
ch_CGCNN_att_test_metrics = {
    "test_acc": test_acc,
    "test_f1": test_f1,
    "test_auc": test_auc
}
print(f"\nTesting {ch_CGCNN_att.__class__.__name__} on test set...\nTest Accuracy: {test_acc:.4f} | Test F1: {test_f1:.4f} | Test AUC: {test_auc:.4f}")

In [ ]:
from torch_geometric.nn import MessagePassing

class CGCNNConv(MessagePassing):
    def __init__(self, in_channels, edge_dim):
        super().__init__(aggr='add')
        self.lin = nn.Linear(in_channels*2 + edge_dim, in_channels)
        self.bn = nn.BatchNorm1d(in_channels)

    def forward(self, x, edge_index, edge_attr):
        return self.propagate(edge_index, x=x, edge_attr=edge_attr)

    def message(self, x_i, x_j, edge_attr):
        z = torch.cat([x_i, x_j, edge_attr], dim=1)
        return F.relu(self.lin(z))

    def update(self, aggr_out):
        return self.bn(aggr_out)

In [ ]:
class CharlesCGCNN2(nn.Module):
    def __init__(self, node_dim, edge_dim, global_dim):
        super().__init__()

        self.embedding = nn.Linear(node_dim, 64)

        self.conv1 = CGCNNConv(64, 10)
        self.conv2 = CGCNNConv(64, 10)
        self.conv3 = CGCNNConv(64, 10)

        self.dropout = nn.Dropout(0.3)

        self.fc_global = nn.Linear(16, 64)

        self.fc1 = nn.Linear(128, 64)
        self.fc2 = nn.Linear(64, 2)

    def forward(self, data):

        x = F.relu(self.embedding(data.x))

        x1 = self.conv1(x, data.edge_index, data.edge_attr)
        x2 = self.conv2(x1, data.edge_index, data.edge_attr)
        x3 = self.conv3(x2, data.edge_index, data.edge_attr)

        x = x1 + x2 + x3  # residual connection

        x = global_mean_pool(x, data.batch)

        u = F.relu(self.fc_global(data.u))

        x = torch.cat([x, u], dim=1)

        x = F.relu(self.fc1(x))
        x = self.dropout(x)

        x = self.fc2(x)

        return x

ch_CGCNN2 = CharlesCGCNN2(node_dim=node_dim, edge_dim=edge_dim, global_dim=global_dim).to(device)
loss_fn = nn.CrossEntropyLoss()

ch_CGCNN2_metrics = train_model(ch_CGCNN2, train_loader, val_loader, loss_fn, epochs=50)
test_acc, test_f1, test_auc = evaluate_model(ch_CGCNN2, test_loader)
ch_CGCNN2_test_metrics = {
    "test_acc": test_acc,
    "test_f1": test_f1,
    "test_auc": test_auc
}
print(f"\nTesting {ch_CGCNN2.__class__.__name__} on test set...\nTest Accuracy: {test_acc:.4f} | Test F1: {test_f1:.4f} | Test AUC: {test_auc:.4f}")

In [ ]:
class CharlesAttentionCGCNN(nn.Module):
    def __init__(self):
        super().__init__()

        # Embed atom features
        self.embedding = nn.Linear(8, 64)

        # Attention layers
        self.gat1 = GATConv(64, 64, heads=2, concat=False)
        self.gat2 = GATConv(64, 64, heads=2, concat=False)

        self.dropout = nn.Dropout(0.3)

        self.fc_global = nn.Linear(16, 64)

        self.fc1 = nn.Linear(128, 64)
        self.fc2 = nn.Linear(64, 2)

    def forward(self, data, return_attention=False):

        x = F.relu(self.embedding(data.x))

        x, attn1 = self.gat1(x, data.edge_index, return_attention_weights=True)
        x = F.relu(x)

        x, attn2 = self.gat2(x, data.edge_index, return_attention_weights=True)

        x = global_mean_pool(x, data.batch)

        u = F.relu(self.fc_global(data.u))

        x = torch.cat([x, u], dim=1)

        x = F.relu(self.fc1(x))
        x = self.dropout(x)

        out = self.fc2(x)

        if return_attention:
            return out, attn1, attn2

        return out

In [ ]:
import matplotlib.pyplot as plt

# Convert to CPU
attn = attention_weights.detach().cpu().numpy()

plt.figure(figsize=(6,4))
plt.hist(attn, bins=30)
plt.title("Attention Weights Distribution")
plt.xlabel("Attention Weight")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# Get one sample from test set
model.eval()

sample = next(iter(test_loader))
sample = sample.to(device)

out, attn1, attn2 = model(sample, return_attention=True)

edge_index, attention_weights = attn1

# Move to CPU
edge_index = edge_index.cpu().numpy()
attn = attention_weights.detach().cpu().numpy()
pos = sample.pos.cpu().numpy()

In [ ]:
# Initialize importance for each atom
num_nodes = pos.shape[0]
node_importance = np.zeros(num_nodes)

# Accumulate attention scores from edges → nodes
for i, (src, dst) in enumerate(edge_index.T):
    # Aggregate attention weights from multiple heads for the current edge
    # Taking the mean across attention heads
    attention_score_for_edge = attn[i].mean()
    node_importance[src] += attention_score_for_edge
    node_importance[dst] += attention_score_for_edge

# Normalize (0 to 1)
node_importance = node_importance / node_importance.max()

In [ ]:
# Initialize importance for each atom
num_nodes = pos.shape[0]
node_importance = np.zeros(num_nodes)

# Accumulate attention scores from edges → nodes
for i, (src, dst) in enumerate(edge_index.T):
    # Aggregate attention weights from multiple heads for the current edge
    attention_score_for_edge = attn[i].mean() # Take the mean of attention weights across heads
    node_importance[src] += attention_score_for_edge
    node_importance[dst] += attention_score_for_edge

# Normalize (0 to 1)
node_importance = node_importance / node_importance.max()

In [ ]:
# =========================================================
# Improved CGCNN with Attention Pooling
# =========================================================
class CharlesCGCNN(nn.Module):
    def __init__(self):
        super().__init__()
        hidden = 128

        self.embed = nn.Linear(NODE_DIM, hidden)

        self.conv1 = CGConv(hidden, dim=43)
        self.conv2 = CGConv(hidden, dim=43)
        self.conv3 = CGConv(hidden, dim=43)

        self.pool = GlobalAttention(
            gate_nn=nn.Sequential(nn.Linear(hidden, 1))
        )

        self.fc1 = nn.Linear(hidden + len(GLOBAL_COLS) + 3, 128)
        self.fc2 = nn.Linear(128, 2)

        self.dropout = nn.Dropout(0.3)

    def forward(self, data):
        x, ei, ea, batch, u = data.x, data.edge_index, data.edge_attr, data.batch, data.u.squeeze(1)

        x = F.relu(self.embed(x))
        x = F.relu(self.conv1(x, ei, ea))
        x = self.dropout(x)
        x = F.relu(self.conv2(x, ei, ea))
        x = self.dropout(x)
        x = F.relu(self.conv3(x, ei, ea))

        x = self.pool(x, batch)

        x = torch.cat([x, u], dim=1)

        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)

model = CharlesCGCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

best_auc = 0

for epoch in range(1, 101):
    model.train()
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch)
        loss = criterion(out, batch.y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        optimizer.step()

    # Validation
    model.eval()
    preds, probs, labels = [], [], []

    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            out = model(batch)
            preds.extend(out.argmax(1).cpu().numpy())
            probs.extend(torch.softmax(out,1)[:,1].cpu().numpy())
            labels.extend(batch.y.cpu().numpy())

    auc = roc_auc_score(labels, probs)
    acc = accuracy_score(labels, preds)

    print(f"Epoch {epoch} | Acc {acc:.3f} | AUC {auc:.3f}")

    if auc > best_auc:
        best_auc = auc
        torch.save(model.state_dict(), os.path.join(BASE_DIR, "best_cgcnn.pth"))
model.load_state_dict(torch.load(os.path.join(BASE_DIR, "best_cgcnn.pth")))
model.eval()

preds, probs, labels = [], [], []

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        out = model(batch)
        preds.extend(out.argmax(1).cpu().numpy())
        probs.extend(torch.softmax(out,1)[:,1].cpu().numpy())
        labels.extend(batch.y.cpu().numpy())

print("Accuracy:", accuracy_score(labels, preds))
print("F1:", f1_score(labels, preds))
print("AUC:", roc_auc_score(labels, probs))

# CharlesMEGNet (Improved Version)

Goal:
- Use physics-based graph representation (Voronoi neighbors)
- Improve over CGCNN

Enhancements:
- Gaussian edge features
- Better global feature integration
- Deeper architecture
- Dropout regularization

Reference:
- Chen et al. (2019) — MEGNet

In [ ]:
from pymatgen.analysis.local_env import VoronoiNN

voronoi = VoronoiNN()

def build_megnet_graph(row):

    structure = Structure.from_file('/content/drive/MyDrive/Project/' + row['cif'])

    # Node features
    x = []
    for site in structure:
        el = site.specie
        x.append([
            el.Z,
            el.X or 0,
            el.atomic_mass or 0,
            el.row or 0,
            el.group or 0
        ])
    x = torch.tensor(x, dtype=torch.float)

    pos = torch.tensor(structure.cart_coords, dtype=torch.float)

    edge_index = []
    distances = []

    for i in range(len(structure)):
        neighbors = voronoi.get_nn_info(structure, i)

        for n in neighbors:
            j = n['site_index']
            d = structure.get_distance(i, j)

            edge_index.append([i, j])
            distances.append(d)

    edge_index = torch.tensor(edge_index).t().contiguous()
    distances = torch.tensor(distances, dtype=torch.float)

    # Gaussian edge features
    centers = torch.linspace(0, 6, 10)
    edge_attr = torch.exp(-((distances.unsqueeze(1) - centers)**2))

    # Global features
    u = torch.tensor(row[global_cols].values, dtype=torch.float).unsqueeze(0)

    y = torch.tensor([row['label']], dtype=torch.long)

    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr,
                pos=pos, u=u, y=y)
class CharlesMEGNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.node_embed = nn.Linear(8, 64) # Changed from 5 to 8 to match input node features
        self.edge_embed = nn.Linear(10, 32)
        self.global_embed = nn.Linear(16, 64)

        self.meg1 = MEGNetLayer(64, 32, 64)
        self.meg2 = MEGNetLayer(64, 32, 64)
        self.meg3 = MEGNetLayer(64, 32, 64)

        self.dropout = nn.Dropout(0.3)

        self.fc = nn.Linear(64, 2)

    def forward(self, data):

        x = F.relu(self.node_embed(data.x))
        e = F.relu(self.edge_embed(data.edge_attr))
        u = F.relu(self.global_embed(data.u))

        x, e, u = self.meg1(x, data.edge_index, e, u)
        x, e, u = self.meg2(x, data.edge_index, e, u)
        x, e, u = self.meg3(x, data.edge_index, e, u)

        u = self.dropout(u)

        out = self.fc(u)

        return out
import torch
import torch.nn as nn
import torch.nn.functional as F

# Minimal placeholder for MEGNetLayer to resolve NameError
# A full MEGNetLayer implementation is more complex, involving message passing and detailed update networks.
class MEGNetLayer(nn.Module):
    def __init__(self, node_dim, edge_dim, global_dim):
        super().__init__()
        # Placeholder linear layers to match input/output dimensions
        self.node_lin = nn.Linear(node_dim, node_dim)
        self.edge_lin = nn.Linear(edge_dim, edge_dim)
        self.global_lin = nn.Linear(global_dim, global_dim)

    def forward(self, x, edge_index, edge_attr, u):
        # Simply pass through linear layers for now
        # A real MEGNet layer would involve message passing and complex updates
        x_out = self.node_lin(x)
        e_out = self.edge_lin(edge_attr)
        u_out = self.global_lin(u)
        return x_out, e_out, u_out

model = CharlesMEGNet().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5
)

criterion = nn.CrossEntropyLoss()

for epoch in range(80):

    model.train()
    total_loss = 0

    for batch in train_loader:
        batch = batch.to(device)

        optimizer.zero_grad()

        out = model(batch)
        loss = criterion(out, batch.y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch} Loss: {total_loss:.4f}")
model.eval()

preds, labels = [], []

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        out = model(batch)
        pred = out.argmax(dim=1)

        preds.extend(pred.cpu().numpy())
        labels.extend(batch.y.cpu().numpy())

from sklearn.metrics import accuracy_score
print("MEGNet Accuracy:", accuracy_score(labels, preds))

In [ ]:
# MEGNet Graph Construction


CUTOFF = 5.0

def build_megnet_graph(row):

    structure = Structure.from_file(os.path.join(PROJECT_PATH, row['cif']))

    # Node features
    node_features = []

    for site in structure:
        el = site.specie

        node_features.append([
            el.Z,
            el.X if el.X else 0,
            el.atomic_radius if el.atomic_radius else 0,
            el.row,
            el.group
        ])

    x = torch.tensor(node_features, dtype=torch.float)

    # Positions
    pos = torch.tensor(structure.cart_coords, dtype=torch.float)

    # Edges
    edge_index = radius_graph(pos, r=CUTOFF, loop=False)

    src, dst = edge_index

    distances = torch.norm(pos[src] - pos[dst], dim=1)

    # RBF expansion
    centers = torch.linspace(0, CUTOFF, 16)
    edge_attr = torch.exp(-((distances.unsqueeze(1) - centers)**2))

    # Global state
    u = torch.tensor(row[global_cols].astype(float).values,
                     dtype=torch.float).unsqueeze(0)

    y = torch.tensor([row['label']], dtype=torch.long)

    return Data(x=x, edge_index=edge_index,
                edge_attr=edge_attr, pos=pos,
                u=u, y=y)


# MEGNet Block

class MEGNetBlock(nn.Module):
    def __init__(self, node_dim, edge_dim, global_dim):
        super().__init__()

        # Edge update
        self.edge_mlp = nn.Sequential(
            nn.Linear(node_dim*2 + edge_dim + global_dim, 64),
            nn.SiLU(),
            nn.Linear(64, edge_dim)
        )

        # Node update
        self.node_mlp = nn.Sequential(
            nn.Linear(node_dim + edge_dim + global_dim, 64),
            nn.SiLU(),
            nn.Linear(64, node_dim)
        )

        # Global update
        self.global_mlp = nn.Sequential(
            nn.Linear(global_dim + node_dim + edge_dim, 64),
            nn.SiLU(),
            nn.Linear(64, global_dim)
        )

    def forward(self, x, edge_index, edge_attr, u, batch):

        src, dst = edge_index

        # -----------------
        # Edge update
        # -----------------
        edge_input = torch.cat([
            x[src], x[dst], edge_attr,
            u[batch[src]]
        ], dim=1)

        edge_attr = self.edge_mlp(edge_input)

        # Node update
        agg = torch.zeros_like(x)

        agg.index_add_(0, dst, edge_attr)

        node_input = torch.cat([
            x, agg,
            u[batch]
        ], dim=1)

        x = self.node_mlp(node_input)

        # Global update
        u_input = torch.cat([
            u,
            global_mean_pool(x, batch),
            global_mean_pool(edge_attr, batch[src])
        ], dim=1)

        u = self.global_mlp(u_input)

        return x, edge_attr, u
# Charles MEGNet Model

class CharlesMEGNet(nn.Module):
    def __init__(self):
        super().__init__()

        node_dim = 64
        edge_dim = 64 # Changed from 16 to 64 to match node_dim
        global_dim = len(global_cols)

        # Embeddings
        self.node_embed = nn.Linear(8, node_dim)
        self.edge_embed = nn.Linear(10, edge_dim)
        self.global_embed = nn.Linear(global_dim, global_dim)

        # MEGNet blocks
        self.block1 = MEGNetBlock(node_dim, edge_dim, global_dim)
        self.block2 = MEGNetBlock(node_dim, edge_dim, global_dim)
        self.block3 = MEGNetBlock(node_dim, edge_dim, global_dim)

        # Set2Set pooling (important)
        self.pool = Set2Set(node_dim, processing_steps=3)

        # Final MLP
        self.fc = nn.Sequential(
            nn.Linear(node_dim*2 + global_dim, 64),
            nn.SiLU(),
            nn.Linear(64, 2)
        )

    def forward(self, data):

        x = self.node_embed(data.x)
        e = self.edge_embed(data.edge_attr)
        u = self.global_embed(data.u)

        batch = data.batch

        x, e, u = self.block1(x, data.edge_index, e, u, batch)
        x, e, u = self.block2(x, data.edge_index, e, u, batch)
        x, e, u = self.block3(x, data.edge_index, e, u, batch)

        # Set2Set pooling
        x = self.pool(x, batch)

        # Combine with global state
        x = torch.cat([x, u], dim=1)

        return self.fc(x)

In [ ]:
# CharlesCGCNN definition
class CharlesCGCNN(nn.Module):
    def __init__(self, node_feat_dim=8, edge_feat_dim=1,
                 hidden_dim=64, global_dim=12, num_classes=2):
        super().__init__()
        # Initial embedding of raw atom features
        self.node_emb = nn.Linear(node_feat_dim, hidden_dim)

        # CGConv layers – integer channels, so in_dim = out_dim = hidden_dim
        self.conv1 = CGConv(hidden_dim, dim=edge_feat_dim)
        self.conv2 = CGConv(hidden_dim, dim=edge_feat_dim)

        # Batch normalisation for stable training
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)

        # Classifier: pooled atom embedding + global features
        self.fc = nn.Linear(hidden_dim + global_dim, num_classes)

    def forward(self, data):
        x = data.x
        edge_index = data.edge_index
        edge_attr = data.edge_attr
        batch = data.batch
        u = data.u
        if u.dim() == 3:               # in case u has shape [B,1,12]
            u = u.squeeze(1)

        # Embed atoms
        x = F.relu(self.node_emb(x))    # [N, hidden_dim]

        # First convolution
        x = self.conv1(x, edge_index, edge_attr)
        x = self.bn1(x)
        x = F.relu(x)

        # Second convolution
        x = self.conv2(x, edge_index, edge_attr)
        x = self.bn2(x)
        x = F.relu(x)

        # Global mean pool → graph-level vector
        x = global_mean_pool(x, batch)  # [B, hidden_dim]

        # Concatenate global features
        out = torch.cat([x, u], dim=1)  # [B, hidden_dim+12]
        out = self.fc(out)               # [B, 2]
        return out

torch.manual_seed(42)
charles_cgcnn = CharlesCGCNN(
    node_feat_dim=8,
    edge_feat_dim=1,
    hidden_dim=64,
    global_dim=12,
    num_classes=2
).to(device)

print(charles_cgcnn)
n_params = sum(p.numel() for p in charles_cgcnn.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {n_params:,}")

# Test forward pass with one batch
charles_cgcnn.eval()
with torch.no_grad():
    out = charles_cgcnn(sample_batch.to(device))
print(f"Output shape: {out.shape}  (should be [{BATCH_SIZE}, 2])")

In [ ]:
class CharlesAttentionGNN(nn.Module):
    def __init__(self, node_feat_dim=2, edge_feat_dim=1,
                 hidden_dim=64, heads=4, global_dim=12, num_classes=2):
        super().__init__()
        self.node_emb = nn.Linear(node_feat_dim, hidden_dim)

        # GAT layer 1: multi-head, concatenated
        self.gat1 = GATConv(hidden_dim, hidden_dim,
                            heads=heads, edge_dim=edge_feat_dim, concat=True)
        # GAT layer 2: single head, average
        self.gat2 = GATConv(hidden_dim * heads, hidden_dim,
                            heads=1, concat=False)

        self.bn1 = nn.BatchNorm1d(hidden_dim * heads)
        self.bn2 = nn.BatchNorm1d(hidden_dim)

        self.fc = nn.Linear(hidden_dim + global_dim, num_classes)

    def forward(self, data, return_attention=False):
        x = data.x
        edge_index = data.edge_index
        edge_attr = data.edge_attr
        batch = data.batch
        u = data.u
        if u.dim() == 3:
            u = u.squeeze(1)

        x = F.relu(self.node_emb(x))          # [N, hidden_dim]

        if return_attention:
            x, (att_edge, att_weight) = self.gat1(
                x, edge_index, edge_attr, return_attention_weights=True
            )
        else:
            x = self.gat1(x, edge_index, edge_attr)
        x = F.elu(self.bn1(x))                # [N, hidden_dim*heads]

        x = self.gat2(x, edge_index)
        x = F.elu(self.bn2(x))                # [N, hidden_dim]

        x = global_mean_pool(x, batch)        # [B, hidden_dim]
        out = torch.cat([x, u], dim=1)
        out = self.fc(out)

        if return_attention:
            return out, att_edge, att_weight
        return out

torch.manual_seed(42)
charles_attn = CharlesAttentionGNN(
    node_feat_dim=8, edge_feat_dim=1, # Note: Changed node_feat_dim from 2 to 8 as per build_graph output
    hidden_dim=64, heads=4,
    global_dim=12, num_classes=2 # Note: Changed global_dim to 12 as per build_graph output
).to(device)


# Train CharlesAttentionGNN
attn_optimizer = torch.optim.Adam(charles_attn.parameters(), lr=1e-3, weight_decay=1e-5)
attn_history = {'train_loss': [], 'val_acc': [], 'val_f1': [], 'val_auc': []}

print("Training CharlesAttentionGNN...")
for epoch in range(1, NUM_EPOCHS+1):
    charles_attn.train()
    total_loss = 0.0
    for batch in train_loader:
        batch = batch.to(device)
        attn_optimizer.zero_grad()
        out = charles_attn(batch)
        loss = criterion(out, batch.y)
        loss.backward()
        attn_optimizer.step()
        total_loss += loss.item() * batch.num_graphs
    avg_loss = total_loss / len(train_dataset)

    # Validation
    charles_attn.eval()
    preds, probs, labels = [], [], []
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            out = charles_attn(batch)
            preds.extend(out.argmax(dim=1).cpu().numpy())
            probs.extend(torch.softmax(out, dim=1)[:,1].cpu().numpy())
            labels.extend(batch.y.cpu().numpy())
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, zero_division=0)
    auc = roc_auc_score(labels, probs)

    attn_history['train_loss'].append(avg_loss)
    attn_history['val_acc'].append(acc)
    attn_history['val_f1'].append(f1)
    attn_history['val_auc'].append(auc)

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{NUM_EPOCHS} | Loss: {avg_loss:.4f} | "
              f"Val Acc: {acc:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}")

# Test evaluation
charles_attn.eval()
test_preds_attn, test_probs_attn, test_labels_attn = [], [], []
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        out = charles_attn(batch)
        test_preds_attn.extend(out.argmax(dim=1).cpu().numpy())
        test_probs_attn.extend(torch.softmax(out, dim=1)[:,1].cpu().numpy())
        test_labels_attn.extend(batch.y.cpu().numpy())

print("\n=== CharlesAttentionGNN – TEST SET ===")
print(f"  Accuracy : {accuracy_score(test_labels_attn, test_preds_attn):.4f}")
print(f"  F1-score : {f1_score(test_labels_attn, test_preds_attn):.4f}")
print(f"  ROC-AUC  : {roc_auc_score(test_labels_attn, test_probs_attn):.4f}")

# Save model
torch.save(charles_attn.state_dict(), os.path.join(BASE_DIR, 'Charles_AttentionGNN.pth'))

In [ ]:
# Train CharlesMEGNet
meg_optimizer = torch.optim.Adam(charles_megnet.parameters(), lr=1e-3, weight_decay=1e-5)
meg_history = {'train_loss': [], 'val_acc': [], 'val_f1': [], 'val_auc': []}

print("Training CharlesMEGNet...")
for epoch in range(1, NUM_EPOCHS+1):
    charles_megnet.train()
    total_loss = 0.0
    for batch in train_loader:
        batch = batch.to(device)
        meg_optimizer.zero_grad()
        out = charles_megnet(batch)
        loss = criterion(out, batch.y)
        loss.backward()
        meg_optimizer.step()
        total_loss += loss.item() * batch.num_graphs
    avg_loss = total_loss / len(train_dataset)

    # Validation
    charles_megnet.eval()
    preds, probs, labels = [], [], []
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            out = charles_megnet(batch)
            preds.extend(out.argmax(dim=1).cpu().numpy())
            probs.extend(torch.softmax(out, dim=1)[:,1].cpu().numpy())
            labels.extend(batch.y.cpu().numpy())
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, zero_division=0)
    auc = roc_auc_score(labels, probs)

    meg_history['train_loss'].append(avg_loss)
    meg_history['val_acc'].append(acc)
    meg_history['val_f1'].append(f1)
    meg_history['val_auc'].append(auc)

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{NUM_EPOCHS} | Loss: {avg_loss:.4f} | "
              f"Val Acc: {acc:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}")

# Test evaluation
charles_megnet.eval()
test_preds_meg, test_probs_meg, test_labels_meg = [], [], []
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        out = charles_megnet(batch)
        test_preds_meg.extend(out.argmax(dim=1).cpu().numpy())
        test_probs_meg.extend(torch.softmax(out, dim=1)[:,1].cpu().numpy())
        test_labels_meg.extend(batch.y.cpu().numpy())

print("\n=== CharlesMEGNet – TEST SET ===")
print(f"  Accuracy : {accuracy_score(test_labels_meg, test_preds_meg):.4f}")
print(f"  F1-score : {f1_score(test_labels_meg, test_preds_meg):.4f}")
print(f"  ROC-AUC  : {roc_auc_score(test_labels_meg, test_probs_meg):.4f}")
print(classification_report(test_labels_meg, test_preds_meg,
                            target_names=['Trivial', 'Topological']))

# Save model
torch.save(charles_megnet.state_dict(), os.path.join(BASE_DIR, 'Charles_MEGNet.pth'))

In [ ]:
#CharlesAttentionGNN definition
class CharlesAttentionGNN(nn.Module):
    def __init__(self, node_feat_dim=2, edge_feat_dim=1,
                 hidden_dim=64, heads=4, global_dim=12, num_classes=2):
        super().__init__()
        self.node_emb = nn.Linear(node_feat_dim, hidden_dim)

        # GAT layer 1: multi-head, concatenated
        self.gat1 = GATConv(hidden_dim, hidden_dim,
                            heads=heads, edge_dim=edge_feat_dim, concat=True)
        # GAT layer 2: single head, average
        self.gat2 = GATConv(hidden_dim * heads, hidden_dim,
                            heads=1, concat=False)

        self.bn1 = nn.BatchNorm1d(hidden_dim * heads)
        self.bn2 = nn.BatchNorm1d(hidden_dim)

        self.fc = nn.Linear(hidden_dim + global_dim, num_classes)

    def forward(self, data, return_attention=False):
        x = data.x
        edge_index = data.edge_index
        edge_attr = data.edge_attr
        batch = data.batch
        u = data.u
        if u.dim() == 3:
            u = u.squeeze(1)

        x = F.relu(self.node_emb(x))          # [N, hidden_dim]

        if return_attention:
            x, (att_edge, att_weight) = self.gat1(
                x, edge_index, edge_attr, return_attention_weights=True
            )
        else:
            x = self.gat1(x, edge_index, edge_attr)
        x = F.elu(self.bn1(x))                # [N, hidden_dim*heads]

        x = self.gat2(x, edge_index)
        x = F.elu(self.bn2(x))                # [N, hidden_dim]

        x = global_mean_pool(x, batch)        # [B, hidden_dim]
        out = torch.cat([x, u], dim=1)
        out = self.fc(out)

        if return_attention:
            return out, att_edge, att_weight
        return out

torch.manual_seed(42)
charles_attn = CharlesAttentionGNN(
    node_feat_dim=2, edge_feat_dim=1,
    hidden_dim=64, heads=4,
    global_dim=N_GLOBAL, num_classes=2
).to(device)

print(charles_attn)

In [ ]:
# CharlesMEGNet definition
class CharlesMEGNet(nn.Module):
    def __init__(self, node_feat_dim=2, edge_feat_dim=1,
                 global_dim=12, hidden_dim=64, n_blocks=3, num_classes=2):
        super().__init__()
        self.node_emb = nn.Linear(node_feat_dim, hidden_dim)
        self.edge_emb = nn.Linear(edge_feat_dim, hidden_dim)

        self.blocks = nn.ModuleList([
            MEGNetBlock(hidden_dim, hidden_dim,
                        global_dim if i==0 else hidden_dim,
                        hidden_dim)
            for i in range(n_blocks)
        ])

        self.pool = Set2Set(hidden_dim, processing_steps=3)
        self.fc = nn.Linear(2*hidden_dim + hidden_dim, num_classes)  # 2*hidden from Set2Set + final global

    def forward(self, data):
        x = data.x
        edge_index = data.edge_index
        edge_attr = data.edge_attr
        u = data.u
        batch = data.batch
        if u.dim() == 3:
            u = u.squeeze(1)

        x = F.relu(self.node_emb(x))
        edge_attr = F.relu(self.edge_emb(edge_attr))

        for block in self.blocks:
            x, edge_attr, u = block(x, edge_index, edge_attr, u, batch)

        x_pooled = self.pool(x, batch)          # [B, 2*hidden]
        out = torch.cat([x_pooled, u], dim=1)   # [B, 2*hidden + hidden]
        out = self.fc(out)
        return out

torch.manual_seed(42)
charles_megnet = CharlesMEGNet(
    node_feat_dim=2, edge_feat_dim=1,
    global_dim=N_GLOBAL, hidden_dim=64, n_blocks=3, num_classes=2
).to(device)

print(charles_megnet)
n_params_meg = sum(p.numel() for p in charles_megnet.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params_meg:,}")

In [ ]:
# MEGNet block
from torch_geometric.utils import scatter

class MEGNetBlock(nn.Module):
    def __init__(self, node_dim, edge_dim, global_dim, hidden):
        super().__init__()
        self.edge_mlp = nn.Sequential(
            nn.Linear(edge_dim + 2*node_dim + global_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden)
        )
        self.node_mlp = nn.Sequential(
            nn.Linear(node_dim + hidden + global_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden)
        )
        self.global_mlp = nn.Sequential(
            nn.Linear(global_dim + hidden + hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden)
        )

    def forward(self, x, edge_index, edge_attr, u, batch):
        src, dst = edge_index
        # Edge update
        u_edge = u[batch[src]]
        edge_input = torch.cat([edge_attr, x[src], x[dst], u_edge], dim=1)
        edge_attr_new = self.edge_mlp(edge_input)

        # Node update
        agg_edges = scatter(edge_attr_new, dst,
                            dim=0, dim_size=x.size(0), reduce='mean')
        u_node = u[batch]
        node_input = torch.cat([x, agg_edges, u_node], dim=1)
        x_new = self.node_mlp(node_input)

        # Global update
        mean_nodes = scatter(x_new, batch,
                             dim=0, dim_size=u.size(0), reduce='mean')
        mean_edges = scatter(edge_attr_new, batch[src],
                             dim=0, dim_size=u.size(0), reduce='mean')
        global_input = torch.cat([u, mean_nodes, mean_edges], dim=1)
        u_new = self.global_mlp(global_input)

        return x_new, edge_attr_new, u_new

In [ ]:
# CGCNN Layer

from torch_geometric.nn import MessagePassing

class CGCNNConv(MessagePassing):
    def __init__(self, in_channels, edge_dim):
        super().__init__(aggr='add')
        self.lin = nn.Linear(in_channels*2 + edge_dim, in_channels)
        self.bn = nn.BatchNorm1d(in_channels)

    def forward(self, x, edge_index, edge_attr):
        return self.propagate(edge_index, x=x, edge_attr=edge_attr)

    def message(self, x_i, x_j, edge_attr):
        z = torch.cat([x_i, x_j, edge_attr], dim=1)
        return F.relu(self.lin(z))

    def update(self, aggr_out):
        return self.bn(aggr_out)

In [ ]:
# CharlesCGCNN Model

class CharlesCGCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.embedding = nn.Linear(8, 64)

        self.conv1 = CGCNNConv(64, 10)
        self.conv2 = CGCNNConv(64, 10)
        self.conv3 = CGCNNConv(64, 10)

        self.fc_global = nn.Linear(len(global_cols), 64)

        self.fc1 = nn.Linear(128, 64)
        self.fc2 = nn.Linear(64, 2)

        self.dropout = nn.Dropout(0.3)

    def forward(self, data):

        x = F.relu(self.embedding(data.x))

        x1 = self.conv1(x, data.edge_index, data.edge_attr)
        x2 = self.conv2(x1, data.edge_index, data.edge_attr)
        x3 = self.conv3(x2, data.edge_index, data.edge_attr)

        x = x1 + x2 + x3

        x = global_mean_pool(x, data.batch)

        u = F.relu(self.fc_global(data.u))

        x = torch.cat([x, u], dim=1)

        x = F.relu(self.fc1(x))
        x = self.dropout(x)

        return self.fc2(x)
    

model = CharlesCGCNN().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

criterion = nn.CrossEntropyLoss()

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=5
)
epochs = 50
best_acc = 0

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for batch in train_loader:
        batch = batch.to(device)

        optimizer.zero_grad()
        out = model(batch)

        loss = criterion(out, batch.y)
        loss.backward()

        optimizer.step()
        total_loss += loss.item()

    # Validation
    model.eval()
    preds, labels = [], []

    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            out = model(batch)

            pred = out.argmax(dim=1)

            preds.extend(pred.cpu().numpy())
            labels.extend(batch.y.cpu().numpy())

    acc = accuracy_score(labels, preds)
    scheduler.step(acc)

    print(f"Epoch {epoch} | Loss: {total_loss:.4f} | Val Acc: {acc:.4f}")

    if acc > best_acc:
        torch.save(model.state_dict(),
                   os.path.join(PROJECT_PATH, "best_model.pth"))
        best_acc = acc

In [ ]:
# Attention-based CGCNN Layer

from torch_geometric.nn import MessagePassing

class AttentionCGCNNConv(MessagePassing):
    def __init__(self, in_channels, edge_dim):
        super().__init__(aggr='add')

        self.lin = nn.Linear(in_channels*2 + edge_dim, in_channels)

        # Attention mechanism
        self.att = nn.Linear(in_channels*2 + edge_dim, 1)

        self.bn = nn.BatchNorm1d(in_channels)

    def forward(self, x, edge_index, edge_attr):
        return self.propagate(edge_index, x=x, edge_attr=edge_attr)

    def message(self, x_i, x_j, edge_attr):

        z = torch.cat([x_i, x_j, edge_attr], dim=1)

        # Attention score
        alpha = torch.sigmoid(self.att(z))

        # Store for interpretability
        self._alpha = alpha

        return alpha * F.relu(self.lin(z))

    def update(self, aggr_out):
        return self.bn(aggr_out)

In [ ]:
# Charles Attention CGCNN

class CharlesAttentionCGCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.embedding = nn.Linear(8, 64)

        self.conv1 = AttentionCGCNNConv(64, 10)
        self.conv2 = AttentionCGCNNConv(64, 10)
        self.conv3 = AttentionCGCNNConv(64, 10)

        self.fc_global = nn.Linear(len(global_cols), 64)

        self.fc1 = nn.Linear(128, 64)
        self.fc2 = nn.Linear(64, 2)

        self.dropout = nn.Dropout(0.3)

    def forward(self, data):

        x = F.relu(self.embedding(data.x))

        x1 = self.conv1(x, data.edge_index, data.edge_attr)
        x2 = self.conv2(x1, data.edge_index, data.edge_attr)
        x3 = self.conv3(x2, data.edge_index, data.edge_attr)

        x = x1 + x2 + x3

        # Save attention weights (last layer)
        self.att_weights = self.conv3._alpha

        x = global_mean_pool(x, data.batch)

        u = F.relu(self.fc_global(data.u))

        x = torch.cat([x, u], dim=1)

        x = F.relu(self.fc1(x))
        x = self.dropout(x)

        return self.fc2(x)

# Model evaluation

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], color='steelblue')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')

axes[1].plot(history['val_acc'], label='Accuracy', color='green')
axes[1].plot(history['val_f1'], label='F1', color='orange')
axes[1].plot(history['val_auc'], label='AUC', color='red')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Score')
axes[1].set_title('Validation Metrics')
axes[1].legend()
axes[1].set_ylim([0.5, 1.05])
plt.tight_layout()
plt.show()

In [ ]:
# Extract attention for one test sample (prefer a TI sample)
charles_attn.eval()

# Find a TI sample in test set
ti_sample = None
for batch in test_loader:
    batch = batch.to(device)
    for i in range(batch.num_graphs):
        if batch.y[i].item() == 1:   # topological
            # Extract single graph (simplest: get one-element batch)
            mask = (batch.batch == i)
            node_mask = mask
            edge_mask = mask[batch.edge_index[0]] | mask[batch.edge_index[1]]
            x_i = batch.x[mask]
            pos_i = batch.pos[mask]
            ei_i = batch.edge_index[:, edge_mask]
            # re-index edges
            node_offset = mask.nonzero(as_tuple=True)[0][0].item()
            ei_i = ei_i - node_offset
            ea_i = batch.edge_attr[edge_mask]
            u_i = batch.u[i:i+1]
            y_i = batch.y[i:i+1]
            ti_sample = Data(x=x_i, edge_index=ei_i, edge_attr=ea_i,
                             u=u_i, y=y_i, pos=pos_i).to(device)
            break
    if ti_sample is not None:
        break

if ti_sample is None:
    print("No TI sample found in test set – using first sample")
    ti_sample = next(iter(test_loader))[0:1].to(device)

# Forward pass with attention
with torch.no_grad():
    _, att_edge, att_weight = charles_attn(ti_sample, return_attention=True)
# att_weight shape [num_edges, num_heads] – average over heads
att_weight = att_weight.mean(dim=1).cpu().numpy()   # [E,]

# Aggregate attention to nodes (sum incoming attention)
node_attn = np.zeros(ti_sample.num_nodes)
edge_idx = att_edge.cpu().numpy()
for e, dst in enumerate(edge_idx[1]):
    node_attn[dst] += att_weight[e]
# Normalise to [0,1]
node_attn = (node_attn - node_attn.min()) / (node_attn.max() - node_attn.min() + 1e-8)

# Plot
coords = ti_sample.pos.cpu().numpy()
Z_vals = ti_sample.x[:,0].cpu().numpy()   # atomic numbers

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
# Scatter plot
sc = axes[0].scatter(coords[:,0], coords[:,1], c=node_attn, cmap='hot',
                     s=200, edgecolors='black')
plt.colorbar(sc, ax=axes[0], label='Attention')
for i, (x, y, z) in enumerate(zip(coords[:,0], coords[:,1], Z_vals)):
    axes[0].annotate(f'Z={int(z)}', (x, y), xytext=(4,4),
                     textcoords='offset points', fontsize=8,
                     color='white' if node_attn[i] > 0.6 else 'black')
axes[0].set_xlabel('x (Å)'); axes[0].set_ylabel('y (Å)')
axes[0].set_title('Attention Heatmap (XY plane)')

# Bar chart
axes[1].barh(range(ti_sample.num_nodes), node_attn, color=plt.cm.hot(node_attn))
axes[1].set_yticks(range(ti_sample.num_nodes))
axes[1].set_yticklabels([f'Z={int(z)}' for z in Z_vals])
axes[1].set_xlabel('Attention Score')
axes[1].set_title('Per‑Atom Attention')
axes[1].invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Wrap model for Explainer (needs (x, edge_index, edge_attr, batch, u) interface)
class CGCNNWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x, edge_index, edge_attr, batch, u):
        # Reconstruct a Data object for the model
        from torch_geometric.data import Data
        data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr,
                    batch=batch, u=u)
        return self.model(data)

wrapped_model = CGCNNWrapper(charles_cgcnn).to(device)
wrapped_model.eval()

# Use one test sample
explainer = Explainer(
    model=wrapped_model,
    algorithm=GNNExplainer(epochs=200),
    explanation_type='model',
    node_mask_type='attributes',      # mask per node feature
    edge_mask_type='object',          # mask per edge
    model_config=dict(
        mode='multiclass_classification',
        task_level='graph',
        return_type='raw'
    ),
)

# Use the same ti_sample from attention (or pick any)
explanation = explainer(
    x=ti_sample.x,
    edge_index=ti_sample.edge_index,
    edge_attr=ti_sample.edge_attr,
    batch=torch.zeros(ti_sample.num_nodes, dtype=torch.long, device=device),
    u=ti_sample.u,
    target=1   # explain prediction for class 1 (topological)
)

node_feat_mask = explanation.node_mask.detach().cpu().numpy()
edge_mask = explanation.edge_mask.detach().cpu().numpy()

print("Node feature importance (Z, electronegativity):",
      node_feat_mask.mean(axis=0))

# Plot edges with high importance
coords = ti_sample.pos.cpu().numpy()
edge_idx = ti_sample.edge_index.cpu().numpy()
e_norm = (edge_mask - edge_mask.min()) / (edge_mask.max() - edge_mask.min() + 1e-8)

plt.figure(figsize=(6,6))
plt.scatter(coords[:,0], coords[:,1], c='lightblue', s=150, edgecolors='k')
for e, (u, v) in enumerate(zip(edge_idx[0], edge_idx[1])):
    if e_norm[e] > 0.1:
        plt.plot([coords[u,0], coords[v,0]], [coords[u,1], coords[v,1]],
                 'r-', lw=3*e_norm[e], alpha=e_norm[e])
for i, (xi, yi) in enumerate(zip(coords[:,0], coords[:,1])):
    plt.annotate(f'Z={int(ti_sample.x[i,0].item())}', (xi, yi),
                 xytext=(3,3), textcoords='offset points', fontsize=8)
plt.xlabel('x (Å)'); plt.ylabel('y (Å)')
plt.title('GNNExplainer – Important Bonds')
plt.show()

In [ ]:
# Extract attention weights

def get_attention(model, loader):

    model.eval()

    all_att = []

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)

            _ = model(batch)

            att = model.att_weights.cpu().numpy()
            all_att.append(att)

    return np.concatenate(all_att)
# Node importance

def compute_node_importance(graph, attention):

    edge_index = graph.edge_index.numpy()

    node_importance = np.zeros(graph.num_nodes)

    for i, (src, dst) in enumerate(edge_index.T):
        node_importance[dst] += attention[i]

    return node_importance
# Attention heatmap

import matplotlib.pyplot as plt

def plot_attention(attention):

    plt.figure(figsize=(6,4))
    plt.hist(attention, bins=50)

    plt.title("Attention Distribution")
    plt.xlabel("Attention Weight")
    plt.ylabel("Frequency")
    plt.show()